In [1]:
%%writefile c_to_f.cu

#include <stdio.h>
#include "cuda_runtime.h"
#include "device_launch_parameters.h"

#define N 5

__global__ void convert(float *celsius, float *fahrenheit)
{
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < N)
    {
        fahrenheit[idx] = (celsius[idx] * 9 / 5) + 32;
    }
}

int main()
{
    float host_celsius[N] = {0, 10, 20, 30, 100};
    float host_fahrenheit[N];

    // device pointers
    float *device_celsius, *device_fahrenheit;

    // allocate memory on device
    cudaMalloc((void **)&device_celsius, N * sizeof(float));
    cudaMalloc((void **)&device_fahrenheit, N * sizeof(float));

    // copy input array from host to device
    cudaMemcpy(device_celsius, host_celsius, N * sizeof(float), cudaMemcpyHostToDevice);

    // launch kernel: one block, N threads (one per element)
    convert<<<1, N>>>(device_celsius, device_fahrenheit);
    cudaDeviceSynchronize();

    // copy result back from device to host
    cudaMemcpy(host_fahrenheit, device_fahrenheit, N * sizeof(float), cudaMemcpyDeviceToHost);

    // display the result
    printf("Celsius:    ");
    for (int i = 0; i < N; i++)
        printf("%.2f ", host_celsius[i]);
    printf("\n");

    printf("Fahrenheit: ");
    for (int i = 0; i < N; i++)
        printf("%.2f ", host_fahrenheit[i]);
    printf("\n");

    // free memory
    cudaFree(device_celsius);
    cudaFree(device_fahrenheit);

    return 0;
}

Writing c_to_f.cu


In [2]:
!nvcc c_to_f.cu -o c_to_f

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [3]:
!./c_to_f

Celsius:    0.00 10.00 20.00 30.00 100.00 
Fahrenheit: 32.00 50.00 68.00 86.00 212.00 
